# Gold — `dbr_caresync.gold.patient_360`

One row per patient. Joins silver patients ↔ hospitals, insurance_providers, and an aggregation of lab_results. Full rebuild on every run (CREATE OR REPLACE TABLE).

In [0]:
# ── Ensure gold schema exists ─────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS dbr_caresync.gold")

# ── Build gold.patient_360 ────────────────────────────────────────────────────
spark.sql("""
CREATE OR REPLACE TABLE dbr_caresync.gold.patient_360 AS
WITH lab_agg AS (
    SELECT
        patient_id,
        COUNT(*)                                                  AS total_tests,
        COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)    AS abnormal_test_count,
        COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)    AS critical_test_count,
        MAX(test_date)                                            AS last_test_date
    FROM dbr_caresync.silver.lab_results
    GROUP BY patient_id
)
SELECT
    p.patient_id,
    p.first_name,
    p.last_name,
    CAST(p.date_of_birth AS DATE)                                                      AS date_of_birth,
    CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
    p.gender,
    p.city,
    p.state,
    p.registered_hospital_id                                                           AS hospital_id,
    h.hospital_name,
    p.insurance_provider_id,
    ip.provider_name                                                                   AS insurance_provider_name,
    COALESCE(la.total_tests,         0)                                                AS total_tests,
    COALESCE(la.abnormal_test_count, 0)                                                AS abnormal_test_count,
    COALESCE(la.critical_test_count, 0)                                                AS critical_test_count,
    la.last_test_date
FROM      dbr_caresync.silver.patients           p
LEFT JOIN dbr_caresync.silver.hospitals           h  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN dbr_caresync.silver.insurance_providers ip ON p.insurance_provider_id  = ip.insurance_provider_id
LEFT JOIN lab_agg                                 la ON p.patient_id             = la.patient_id
""")

print("\u2705 Table created: dbr_caresync.gold.patient_360")
result = spark.table("dbr_caresync.gold.patient_360")
print(f"   Rows    : {result.count():,}")
print(f"   Columns : {len(result.columns)}")
display(result.limit(5))

In [0]:
record_count = result.count()
dbutils.notebook.exit(str(record_count))